# Cartographie prédictive GWS — GaussianProcessRegressor (GPR)

Notebook Google Colab généré à partir du script Python d'origine : **`GaussianProcess.py`**

**Dossier de données attendu sur Google Drive :** `/content/drive/MyDrive/input_data_ML`
(il doit contenir `name_files.txt` et les rasters listés dedans).

**Ordre d'exécution :** lancer les cellules de haut en bas (*Exécution → Tout exécuter*).
Chaque cellule exécute directement son étape, comme le script d'origine (qui n'a pas de fonction `main()`).
Les résultats sont écrits dans `input_data_ML/GWS_PREDICTIONS_GEE_GPR` sur Drive.

Le code du script est repris tel quel. Les seules lignes ajoutées ou modifiées pour Colab
sont repérées par le commentaire `# [COLAB]`.

### Description (en-tête du script d'origine)

```text
# ============================================================
#  CARTOGRAPHIE PREDICTIVE GWS
#  (TRAIN = longue période, TEST = 5% des dates, CLIP GEE, SANS SOUS-ÉCHANTILLONNAGE)
#
#  - Modèle GaussianProcessRegressor (GPR, scikit-learn)
#    sur grille GLDAS (~25 km), avec StandardScaler
#
#  - Variables explicatives (dossiers = ceux de name_files.txt) :
#       * CHIRPS_Clipped        → CHIRPS (précipitations, journalier)
#       * MODIS_LST_Clipped     → LST MODIS (journalier)
#       * MODIS_NDVI_Clipped    → NDVI MODIS (16 jours, étendu à toutes les dates)
#       * MODIS_ET_Clipped      → ET MODIS (~8 jours, étendu à toutes les dates)
#       * ERA5_Qsb_Clipped      → Qsb ERA5 (journalier)
#       * SMAP_RZSM_Clipped     → RZSM SMAP (journalier)
#       * SMAP_SoilM_Clipped    → Surface Soil Moisture SMAP (journalier)
#
#  - Cible :
#       * GLDAS_GWS_Clipped     → GWS (GLDAS, journalier)
#
#  - Features finales :
#       [CHIRPS, LST, NDVI_ext, ET_ext, Qsb, RZSM, SoilM, sinDOY, cosDOY]
#
#  - Rééchantillonnage en respectant correctement les NoData
#  - Remplissage des NaN d'entrée → cartes SANS TROUS
#    à l’intérieur des Regions (asset GEE "Regions")
#  - Clip de toutes les cartes (test + forecast) sur la zone
#    Regions reconstruite à partir des rasters GEE
#
#  ➤ Périodes utilisées :
#    FENÊTRE GLOBALE (pool) : 2021-01-01 → 2025-06-30
#    TEST = 5% des dates tirées aléatoirement dans cette fenêtre
#    FORECAST : 2025-07-01 → 2025-10-30
#
#  NB : les dates réellement utilisées dépendent de l’intersection
#       CHIRPS ∩ LST ∩ GWS ∩ Qsb ∩ RZSM ∩ SoilM.
# ============================================================
```

## A. Préparation de l'environnement Colab

### A.1 Monter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### A.2 Installer les dépendances

`rasterio` n'est pas préinstallé sur Colab (scikit-learn, numpy et joblib le sont).

In [ ]:
!pip install -q rasterio

### A.3 Imports

In [ ]:
import os
import re
import time
import bisect
from datetime import datetime, date

import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF,
    ConstantKernel as C,
    WhiteKernel,
)

# [COLAB] Versions des bibliothèques utilisées
from importlib.metadata import version as _pkg_version
for _pkg in ('numpy', 'scikit-learn', 'rasterio'):
    print(f"{_pkg:13s}: {_pkg_version(_pkg)}")

## 1. Paramètres généraux

`BASE_DIR` pointe vers le dossier `input_data_ML` de votre Drive.

In [ ]:
# ------------------------------------------------------------
# 1. Paramètres généraux
# ------------------------------------------------------------
np.random.seed(42)

# Dossier principal contenant les sous-dossiers listés dans name_files.txt
# [COLAB] Dossier d'entrée sur Google Drive (remplace C:\Users\PC\Downloads\input_data_ML)
BASE_DIR = "/content/drive/MyDrive/input_data_ML"

# Fichier texte listant tous les rasters par dossier
NAME_FILE = os.path.join(BASE_DIR, "name_files.txt")

# [COLAB] Arrêt immédiat si Drive n'est pas monté ou si le dossier est mal placé
# (sinon le dossier de sortie serait créé hors de Drive et perdu en fin de session)
if not os.path.isdir(BASE_DIR):
    raise FileNotFoundError(
        f"Dossier introuvable: {BASE_DIR} -> monter Drive (cellule A.1) et vérifier "
        "que input_data_ML est bien à la racine de 'Mon Drive'."
    )
if not os.path.isfile(NAME_FILE):
    raise FileNotFoundError(f"Fichier introuvable: {NAME_FILE}")

# Sous-dossiers (principalement pour affichage / debug)
CHIRPS_DIR = os.path.join(BASE_DIR, "CHIRPS_Clipped")
LST_DIR    = os.path.join(BASE_DIR, "MODIS_LST_Clipped")
NDVI_DIR   = os.path.join(BASE_DIR, "MODIS_NDVI_Clipped")   # 16 jours
ET_DIR     = os.path.join(BASE_DIR, "MODIS_ET_Clipped")     # ~8 jours
GWS_DIR    = os.path.join(BASE_DIR, "GLDAS_GWS_Clipped")    # cible
QSB_DIR    = os.path.join(BASE_DIR, "ERA5_Qsb_Clipped")     # journalier
RZSM_DIR   = os.path.join(BASE_DIR, "SMAP_RZSM_Clipped")    # journalier
SOILM_DIR  = os.path.join(BASE_DIR, "SMAP_SoilM_Clipped")   # journalier

OUT_DIR = os.path.join(BASE_DIR, "GWS_PREDICTIONS_GEE_GPR")
os.makedirs(OUT_DIR, exist_ok=True)

NODATA_VALUE = -9999.0

# Fraction des dates utilisées pour le test (5 %)
TEST_FRACTION = 0.05

# Pour limiter le coût O(n^3) du GPR (sinon risque de temps/mémoire énormes)
# Mettre à None pour utiliser tous les pixels d'entraînement.
MAX_TRAIN_SAMPLES_GPR = 8000

## 2. Fenêtres temporelles TRAIN / FORECAST

In [ ]:
# ------------------------------------------------------------
# 2. Fenêtres temporelles TRAIN / FORECAST
# ------------------------------------------------------------
# ➤ Fenêtre "pool" d'entraînement : 01/01/2021 → 30/06/2025
TRAIN_START = date(2021, 1, 1)
TRAIN_END   = date(2025, 6, 30)

# ➤ Fenêtre de prévision demandée (FORECAST)
FORECAST_START = datetime(2025, 7, 1)
FORECAST_END   = datetime(2025, 10, 30)

print("=== PARAMÈTRES GÉNÉRAUX ===")
print(" BASE_DIR :", BASE_DIR)
print(" NAME_FILE :", NAME_FILE)
print(" CHIRPS_DIR :", CHIRPS_DIR)
print(" LST_DIR :", LST_DIR)
print(" NDVI_DIR :", NDVI_DIR)
print(" ET_DIR :", ET_DIR)
print(" GWS_DIR (cible) :", GWS_DIR)
print(" QSB_DIR :", QSB_DIR)
print(" RZSM_DIR :", RZSM_DIR)
print(" SOILM_DIR :", SOILM_DIR)
print(" OUT_DIR :", OUT_DIR)
print(" TRAIN_START/END :", TRAIN_START, "→", TRAIN_END)
print(" TEST_FRACTION :", TEST_FRACTION)
print(" FORECAST_START/END :", FORECAST_START.date(), "→", FORECAST_END.date())
print("============================\n")

### 2.1 DATES EXCLUES (LST manquante 2022-10-11 → 2022-10-22)

In [ ]:
# ------------------------------------------------------------
# 2.1. DATES EXCLUES (LST manquante 2022-10-11 → 2022-10-22)
# ------------------------------------------------------------
EXCLUDED_DATES = set()
for day in range(11, 23):
    EXCLUDED_DATES.add(date(2022, 10, day))

print("Dates exclues (LST manquante) :")
for d in sorted(EXCLUDED_DATES):
    print(" -", d)

## 3. Fonctions utilitaires

Adaptation Colab : les entrées de `name_files.txt` écrites sous Windows
(antislashs `\`, chemins absolus `C:\...\input_data_ML\...`) sont converties
automatiquement en chemins Drive par `_win_to_colab_path`.

In [ ]:
# ------------------------------------------------------------
# 3. Fonctions utilitaires
# ------------------------------------------------------------
def extract_date_from_filename(path):
    """
    Extrait YYYYMMDD du nom de fichier (ex : CHIRPS_20200101.tif, SMAP_SoilM__20200110.tif, etc.).
    """
    name = os.path.basename(path)
    m = re.search(r"(\d{8})", name)
    if not m:
        raise ValueError(f"Impossible d'extraire la date depuis : {name}")
    return datetime.strptime(m.group(1), "%Y%m%d").date()


def _win_to_colab_path(p: str) -> str:
    """
    [COLAB] Convertit une entrée écrite sous Windows en chemin utilisable sur Colab (Linux):
    - antislashs -> slashs
    - chemin absolu Windows (C:/.../input_data_ML/xxx) -> xxx (relatif à BASE_DIR)
    Les chemins déjà valides (relatifs, ou absolus sous BASE_DIR) sont inchangés.
    """
    p = str(p).strip().replace("\\", "/")
    if re.match(r"^[A-Za-z]:/", p):
        root_name = os.path.basename(os.path.normpath(BASE_DIR)).lower()
        parts = p.split("/")
        low = [x.lower() for x in parts]
        if root_name in low:
            rest = parts[low.index(root_name) + 1:]
            p = "/".join(rest) if rest else "."
    return p


def parse_name_file(txt_path):
    """
    Parse name_files.txt en un dict:
    {
        "SectionName": [relative/path1.tif, relative/path2.tif, ...],
        ...
    }
    Les chemins sont relatifs à BASE_DIR.
    """
    sections = {}
    current = None
    with open(txt_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith("[") and line.endswith("]"):
                current = line[1:-1].strip()
                sections[current] = []
            else:
                if current is None:
                    continue
                sections[current].append(_win_to_colab_path(line))  # [COLAB] chemin Windows -> Colab
    return sections


def build_date_to_path_dict_from_section(sections, section_name):
    """
    Construit un dict {date: chemin_absolu} pour une section donnée dans name_files.txt
    (par ex. 'CHIRPS_Clipped').
    """
    if section_name not in sections:
        raise RuntimeError(f"Section '{section_name}' absente de {NAME_FILE}")
    d = {}
    for rel_path in sections[section_name]:
        abs_path = os.path.join(BASE_DIR, rel_path)
        abs_path = os.path.normpath(abs_path)
        if not os.path.isfile(abs_path):
            print(f"⚠️ Fichier introuvable (ignoré) : {abs_path}")
            continue
        try:
            date_key = extract_date_from_filename(abs_path)
            d[date_key] = abs_path
        except ValueError as e:
            print("⚠️", e)
    return d


def read_raster_raw(path):
    """
    Lit un raster (bande 1) en float32, SANS modifier les valeurs NoData.
    Retourne (arr, profile, nodata).
    """
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        profile = src.profile
        nodata = src.nodata
    return arr, profile, nodata


def to_float_with_nan(arr, nodata):
    """Convertit le tableau en float32 et remplace nodata → NaN."""
    arr = arr.astype("float32")
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr)
    return arr


def reproject_to_template(
    src_arr,
    src_profile,
    template_profile,
    src_nodata=None,
    dst_nodata=np.nan,
    resampling=Resampling.average,
):
    """
    Rééchantillonne src_arr sur la grille définie par template_profile,
    en respectant les NoData.
    """
    dst_arr = np.full(
        (template_profile["height"], template_profile["width"]),
        dst_nodata,
        dtype="float32",
    )
    kwargs = dict(
        source=src_arr,
        destination=dst_arr,
        src_transform=src_profile["transform"],
        src_crs=src_profile["crs"],
        dst_transform=template_profile["transform"],
        dst_crs=template_profile["crs"],
        dst_nodata=dst_nodata,
        resampling=resampling,
    )
    if src_nodata is not None:
        kwargs["src_nodata"] = src_nodata
    reproject(**kwargs)
    return dst_arr


def fill_nan_with_mean(arr):
    """Remplace les NaN par la moyenne de arr (si possible)."""
    if np.all(np.isnan(arr)):
        return arr
    m = np.nanmean(arr)
    return np.where(np.isnan(arr), m, arr)


def build_prev_date_support(date_dict):
    """
    Construit une fonction get_date_for(target_date) qui retourne
    la dernière date de date_dict <= target_date, ou None si aucune.
    """
    dates_sorted = sorted(date_dict.keys())

    def get_prev(target_date):
        if not dates_sorted:
            return None
        idx = bisect.bisect_right(dates_sorted, target_date) - 1
        if idx < 0:
            return None
        return dates_sorted[idx]

    return get_prev


# --- Masque "Regions" reconstruit à partir des rasters (asset GEE) ---
REGION_MASK = None


def update_region_mask(precip_res, lst_res, ndvi_res, template_profile):
    """
    Met à jour le masque global REGION_MASK en utilisant les rasters
    rééchantillonnés sur la grille GLDAS.
    On suppose que les rasters ont été clipés par l'asset GEE "Regions".
    """
    global REGION_MASK
    base_mask = ~(np.isnan(precip_res) & np.isnan(lst_res) & np.isnan(ndvi_res))  # True = inside
    if REGION_MASK is None:
        REGION_MASK = base_mask
        print("\n🗺️ Masque 'Regions' initial construit à partir des rasters (1er jour).")
    else:
        REGION_MASK = REGION_MASK | base_mask


def get_region_mask(template_profile):
    global REGION_MASK
    if REGION_MASK is None:
        print("⚠️ REGION_MASK non initialisé, on prend toute la grille.")
        REGION_MASK = np.ones(
            (template_profile["height"], template_profile["width"]),
            dtype=bool,
        )
    return REGION_MASK

### Contrôle des entrées de `name_files.txt` (ajout Colab)

Vérifie, avant le calcul long, que chaque entrée du fichier se résout bien en fichier présent sur Drive.
Une section en ❌ signale un chemin à corriger. Lecture seule ; peut prendre une à deux minutes sur Drive.

In [ ]:
# [COLAB] Contrôle rapide : les entrées de name_files.txt se résolvent-elles sur Drive ?
_sections_check = parse_name_file(NAME_FILE)
print(f"{len(_sections_check)} sections lues dans {os.path.basename(NAME_FILE)}\n")

for _sec, _entries in _sections_check.items():
    _missing = [_e for _e in _entries
                if not os.path.isfile(os.path.normpath(os.path.join(BASE_DIR, _e)))]
    _n_ok = len(_entries) - len(_missing)
    _flag = "✅" if (_n_ok > 0 and not _missing) else ("⚠️" if _n_ok > 0 else "❌")
    print(f"{_flag} [{_sec}] {len(_entries)} entrées -> {_n_ok} fichiers | introuvables: {len(_missing)}")
    for _e in _missing[:3]:
        print(f"      ex. introuvable: {os.path.normpath(os.path.join(BASE_DIR, _e))}")

## 4. Index des fichiers par date (via name_files.txt)

In [ ]:
# ------------------------------------------------------------
# 4. Index des fichiers par date (via name_files.txt)
# ------------------------------------------------------------
print("\n=== INDEXATION DES FICHIERS PAR DATE (via name_files.txt) ===")
sections = parse_name_file(NAME_FILE)

chirps_dict = build_date_to_path_dict_from_section(sections, "CHIRPS_Clipped")
lst_dict    = build_date_to_path_dict_from_section(sections, "MODIS_LST_Clipped")
ndvi_dict   = build_date_to_path_dict_from_section(sections, "MODIS_NDVI_Clipped")
et_dict     = build_date_to_path_dict_from_section(sections, "MODIS_ET_Clipped")
gws_dict    = build_date_to_path_dict_from_section(sections, "GLDAS_GWS_Clipped")
qsb_dict    = build_date_to_path_dict_from_section(sections, "ERA5_Qsb_Clipped")
rzsm_dict   = build_date_to_path_dict_from_section(sections, "SMAP_RZSM_Clipped")
soilm_dict  = build_date_to_path_dict_from_section(sections, "SMAP_SoilM_Clipped")

print("Nb dates CHIRPS :", len(chirps_dict))
print("Nb dates LST    :", len(lst_dict))
print("Nb dates NDVI   :", len(ndvi_dict))
print("Nb dates ET     :", len(et_dict))
print("Nb dates GWS    :", len(gws_dict))
print("Nb dates Qsb    :", len(qsb_dict))
print("Nb dates RZSM   :", len(rzsm_dict))
print("Nb dates SoilM  :", len(soilm_dict))

if chirps_dict:
    print("CHIRPS :", min(chirps_dict), "→", max(chirps_dict))
if lst_dict:
    print("LST    :", min(lst_dict), "→", max(lst_dict))
if ndvi_dict:
    print("NDVI   :", min(ndvi_dict), "→", max(ndvi_dict))
if et_dict:
    print("ET     :", min(et_dict), "→", max(et_dict))
if gws_dict:
    print("GWS    :", min(gws_dict), "→", max(gws_dict))
if qsb_dict:
    print("Qsb    :", min(qsb_dict), "→", max(qsb_dict))
if rzsm_dict:
    print("RZSM   :", min(rzsm_dict), "→", max(rzsm_dict))
if soilm_dict:
    print("SoilM  :", min(soilm_dict), "→", max(soilm_dict))

# Fonctions de support (NDVI / ET non journaliers)
get_ndvi_date_for = build_prev_date_support(ndvi_dict)
get_et_date_for   = build_prev_date_support(et_dict)

## 5. Construction des dates TRAIN (pool) et TEST (5% des dates)

In [ ]:
# ------------------------------------------------------------
# 5. Construction des dates TRAIN (pool) et TEST (5% des dates)
# ------------------------------------------------------------
print("\n=== CONSTRUCTION DES DATES TRAIN / TEST (5% des dates pour TEST) ===")

# IMPORTANT :
# -> on intersecte uniquement les variables JOURNALIÈRES
#    (CHIRPS, LST, GWS, Qsb, RZSM, SoilM)
# -> NDVI et ET seront étendus via get_ndvi_date_for / get_et_date_for.

all_common_dates = sorted(
    set(chirps_dict.keys())
    & set(lst_dict.keys())
    & set(gws_dict.keys())
    & set(qsb_dict.keys())
    & set(rzsm_dict.keys())
    & set(soilm_dict.keys())
)

print(
    "Nombre total de dates communes (CHIRPS ∩ LST ∩ GWS ∩ Qsb ∩ RZSM ∩ SoilM) :",
    len(all_common_dates),
)
if not all_common_dates:
    raise RuntimeError("⚠️ Aucune date commune entre les variables journalières.")

# Candidats = toutes les dates dans la fenêtre TRAIN, hors dates exclues
date_candidates = [
    d
    for d in all_common_dates
    if (TRAIN_START <= d <= TRAIN_END) and (d not in EXCLUDED_DATES)
]
if len(date_candidates) == 0:
    raise RuntimeError(
        "⚠️ Aucune date candidate dans la fenêtre TRAIN (après exclusion)."
    )

# Split 95% / 5% sur les dates
train_dates, test_dates = train_test_split(
    date_candidates,
    test_size=TEST_FRACTION,
    random_state=42,
    shuffle=True,
)
train_dates = sorted(train_dates)
test_dates  = sorted(test_dates)

if len(train_dates) == 0:
    raise RuntimeError("⚠️ Aucune date disponible pour l'entraînement après split.")
if len(test_dates) == 0:
    raise RuntimeError("⚠️ Aucune date disponible pour le test après split.")

print(f"\nFenêtre TRAIN (pool) : {TRAIN_START} → {TRAIN_END}")
print(f" → Nombre de dates train : {len(train_dates)}")
print(" → Première date train :", min(train_dates))
print(" → Dernière date train :", max(train_dates))

print(f"\nTEST = 5% des dates dans la même fenêtre ({TRAIN_START} → {TRAIN_END})")
print(f" → Nombre de dates test : {len(test_dates)}")
print(" → Première date test :", min(test_dates))
print(" → Dernière date test :", max(test_dates))

## 6. Construction du dataset (TRAIN + TEST, tous pixels valides)

In [ ]:
# ------------------------------------------------------------
# 6. Construction du dataset (TRAIN + TEST, tous pixels valides)
# ------------------------------------------------------------
print("\n=== CONSTRUCTION DU DATASET (toutes les dates TRAIN + TEST) ===")

X_train_list = []
y_train_list = []
X_test_list  = []
y_test_list  = []

template_profile = None


def build_Xy_for_date(date_key, template_profile, role="train"):
    """
    Construit X_day, y_day pour une date donnée sur la grille GLDAS.
    - role = "train" ou "test"
    - utilise TOUS les pixels dont GWS est défini à l’intérieur de Regions
      (les variables explicatives sont préalablement remplies : pas de trous).
    - Entrées : CHIRPS, LST, NDVI(étendu), ET(étendu), Qsb, RZSM, SoilM, sinDOY, cosDOY
    - Cible : GWS
    """
    # --- NDVI (16 jours) : dernière date NDVI <= date_key
    ndvi_date = get_ndvi_date_for(date_key)
    if ndvi_date is None:
        print(
            f"⏭️ [{role}] {date_key} : aucun NDVI disponible (<= {date_key}), jour ignoré."
        )
        return None, None, template_profile

    # --- ET (~8 jours) : dernière date ET <= date_key
    et_date = get_et_date_for(date_key)
    if et_date is None:
        print(
            f"⏭️ [{role}] {date_key} : aucun ET disponible (<= {date_key}), jour ignoré."
        )
        return None, None, template_profile

    # --- GWS : grille cible GLDAS
    if date_key not in gws_dict:
        print(f"⏭️ [{role}] {date_key} : GWS manquant, jour ignoré.")
        return None, None, template_profile

    gws_raw, gws_prof, gws_nodata = read_raster_raw(gws_dict[date_key])
    gws_arr = to_float_with_nan(gws_raw, gws_nodata)

    if template_profile is None:
        template_profile = gws_prof
        print("\nGrille GLDAS (template) :")
        print(" - CRS :", template_profile["crs"])
        print(" - Taille:", template_profile["width"], "x", template_profile["height"])
        print(" - Résolution approx (deg) :", template_profile["transform"][0])

    # --- Entrées brutes ---
    if (
        date_key not in chirps_dict
        or date_key not in lst_dict
        or date_key not in qsb_dict
        or date_key not in rzsm_dict
        or date_key not in soilm_dict
    ):
        print(
            f"⏭️ [{role}] {date_key} : au moins une variable journalière est manquante, jour ignoré."
        )
        return None, None, template_profile

    precip_raw, precip_prof, precip_nodata = read_raster_raw(chirps_dict[date_key])
    lst_raw,    lst_prof,    lst_nodata    = read_raster_raw(lst_dict[date_key])
    ndvi_raw,   ndvi_prof,   ndvi_nodata   = read_raster_raw(ndvi_dict[ndvi_date])
    et_raw,     et_prof,     et_nodata     = read_raster_raw(et_dict[et_date])
    qsb_raw,    qsb_prof,    qsb_nodata    = read_raster_raw(qsb_dict[date_key])
    rzsm_raw,   rzsm_prof,   rzsm_nodata   = read_raster_raw(rzsm_dict[date_key])
    soilm_raw,  soilm_prof,  soilm_nodata  = read_raster_raw(soilm_dict[date_key])

    # --- Rééchantillonnage sur la grille GLDAS ---
    precip_res = reproject_to_template(
        precip_raw,
        precip_prof,
        template_profile,
        src_nodata=precip_nodata,
        dst_nodata=np.nan,
    )
    lst_res = reproject_to_template(
        lst_raw, lst_prof, template_profile, src_nodata=lst_nodata, dst_nodata=np.nan
    )
    ndvi_res = reproject_to_template(
        ndvi_raw, ndvi_prof, template_profile, src_nodata=ndvi_nodata, dst_nodata=np.nan
    )
    et_res = reproject_to_template(
        et_raw, et_prof, template_profile, src_nodata=et_nodata, dst_nodata=np.nan
    )
    qsb_res = reproject_to_template(
        qsb_raw, qsb_prof, template_profile, src_nodata=qsb_nodata, dst_nodata=np.nan
    )
    rzsm_res = reproject_to_template(
        rzsm_raw, rzsm_prof, template_profile, src_nodata=rzsm_nodata, dst_nodata=np.nan
    )
    soilm_res = reproject_to_template(
        soilm_raw,
        soilm_prof,
        template_profile,
        src_nodata=soilm_nodata,
        dst_nodata=np.nan,
    )

    # --- Masque "Regions" (basé sur CHIRPS/LST/NDVI) ---
    update_region_mask(precip_res, lst_res, ndvi_res, template_profile)
    region_mask = get_region_mask(template_profile)

    # --- Remplissage des NaN des variables d'entrée ---
    precip_f = fill_nan_with_mean(precip_res)
    lst_f    = fill_nan_with_mean(lst_res)
    ndvi_f   = fill_nan_with_mean(ndvi_res)
    et_f     = fill_nan_with_mean(et_res)
    qsb_f    = fill_nan_with_mean(qsb_res)
    rzsm_f   = fill_nan_with_mean(rzsm_res)
    soilm_f  = fill_nan_with_mean(soilm_res)

    # --- Masque des pixels valides pour l'échantillon ---
    mask_valid = (~np.isnan(gws_arr)) & region_mask
    if not np.any(mask_valid):
        print(f"⏭️ [{role}] {date_key} : aucun pixel valide, jour ignoré.")
        return None, None, template_profile

    # --- DOY (sin/cos) ---
    doy = date_key.timetuple().tm_yday
    sin_doy = np.sin(2 * np.pi * doy / 365.0)
    cos_doy = np.cos(2 * np.pi * doy / 365.0)

    n_pixels = np.count_nonzero(mask_valid)
    sin_plane = np.full(n_pixels, sin_doy, dtype="float32")
    cos_plane = np.full(n_pixels, cos_doy, dtype="float32")

    # --- Construction de X et y sur les pixels valides ---
    X_day = np.stack(
        [
            precip_f[mask_valid],
            lst_f[mask_valid],
            ndvi_f[mask_valid],
            et_f[mask_valid],
            qsb_f[mask_valid],
            rzsm_f[mask_valid],
            soilm_f[mask_valid],
            sin_plane,
            cos_plane,
        ],
        axis=1,
    )
    y_day = gws_arr[mask_valid]

    # Filtrage final : enlever toutes les lignes qui contiennent encore des NaN
    row_mask = (~np.isnan(y_day)) & (~np.isnan(X_day).any(axis=1))
    if not np.any(row_mask):
        print(
            f"⏭️ [{role}] {date_key} : tous les pixels ont au moins une NaN dans les features, jour ignoré."
        )
        return None, None, template_profile

    X_day = X_day[row_mask]
    y_day = y_day[row_mask]

    print(
        f" ✔️ [{role}] {date_key} : {X_day.shape[0]} pixels utilisés "
        "(GWS défini dans Regions, features finies)."
    )
    return X_day, y_day, template_profile


# --- Boucle TRAIN ---
for d in train_dates:
    X_day, y_day, template_profile = build_Xy_for_date(d, template_profile, role="train")
    if X_day is None:
        continue
    X_train_list.append(X_day)
    y_train_list.append(y_day)

# --- Boucle TEST ---
for d in test_dates:
    X_day, y_day, template_profile = build_Xy_for_date(d, template_profile, role="test")
    if X_day is None:
        continue
    X_test_list.append(X_day)
    y_test_list.append(y_day)

if len(X_train_list) == 0:
    raise RuntimeError("⚠️ Aucun pixel d'entraînement valide n'a été trouvé.")
if len(X_test_list) == 0:
    raise RuntimeError("⚠️ Aucun pixel de test valide n'a été trouvé.")

X_train = np.vstack(X_train_list).astype("float32")
y_train = np.concatenate(y_train_list).astype("float32")
X_test  = np.vstack(X_test_list).astype("float32")
y_test  = np.concatenate(y_test_list).astype("float32")

print(
    f"\nNombre total de pixels d'entraînement (toutes dates TRAIN) : {X_train.shape[0]}"
)
print(f"Nombre total de pixels de test (toutes dates TEST) : {X_test.shape[0]}")
print(
    "Exemple de feature train "
    "(CHIRPS, LST, NDVI_ext, ET_ext, Qsb, RZSM, SoilM, sinDOY, cosDOY) :"
)
print(" ", X_train[0])

## 7. Modèle : GaussianProcessRegressor (GPR) + StandardScaler

In [ ]:
# ------------------------------------------------------------
# 7. Modèle : GaussianProcessRegressor (GPR) + StandardScaler
# ------------------------------------------------------------
print("\n=== ENTRAÎNEMENT DU MODÈLE GAUSSIANPROCESSREGRESSOR (toutes dates TRAIN) ===")

# Option : sous-échantillonnage du set train pour limiter le coût O(n^3)
if (MAX_TRAIN_SAMPLES_GPR is not None) and (X_train.shape[0] > MAX_TRAIN_SAMPLES_GPR):
    print(
        f"\n⚠️ Sous-échantillonnage du set TRAIN pour GPR : "
        f"{X_train.shape[0]} → {MAX_TRAIN_SAMPLES_GPR} pixels"
    )
    idx_sub = np.random.choice(
        X_train.shape[0], MAX_TRAIN_SAMPLES_GPR, replace=False
    )
    X_train_used = X_train[idx_sub]
    y_train_used = y_train[idx_sub]
else:
    X_train_used = X_train
    y_train_used = y_train

# Split interne pour validation (sur les pixels d'entraînement utilisés)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_used,
    y_train_used,
    test_size=0.2,  # 20 % des pixels pour la validation
    random_state=42,
    shuffle=True,
)
print(f"Taille X_tr : {X_tr.shape}, X_val : {X_val.shape}")

# Définition du noyau GPR
# ConstantKernel * RBF + WhiteKernel (bruit)
n_features = X_train.shape[1]
kernel = (
    C(1.0, (1e-3, 1e3))
    * RBF(length_scale=np.ones(n_features), length_scale_bounds=(1e-2, 1e3))
    + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-5, 1e2))
)

gpr = GaussianProcessRegressor(
    kernel=kernel,
    alpha=0.0,          # bruit explicite géré par WhiteKernel
    normalize_y=True,   # normalisation interne de y
    n_restarts_optimizer=2,
    random_state=42,
)

# Pipeline : StandardScaler + GPR
gpr_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("gpr", gpr),
    ]
)

print("\n🔄 Entraînement GaussianProcessRegressor (train)...")
t0 = time.time()
gpr_model.fit(X_tr, y_tr)
t1 = time.time()
training_time_ms = (t1 - t0) * 1000.0

# --- Perf TRAIN sur le sous-ensemble X_tr ---
y_tr_pred = gpr_model.predict(X_tr)
r2_tr   = r2_score(y_tr, y_tr_pred)
rmse_tr = np.sqrt(mean_squared_error(y_tr, y_tr_pred))
mae_tr  = mean_absolute_error(y_tr, y_tr_pred)

# --- Perf VALIDATION sur X_val ---
y_val_pred = gpr_model.predict(X_val)
r2_val   = r2_score(y_val, y_val_pred)
rmse_val = np.sqrt(mean_squared_error(y_val, y_val_pred))
mae_val  = mean_absolute_error(y_val, y_val_pred)

# --- Perf TEST (sur X_test / y_test, dates test 5 %) ---
y_pred_sample = gpr_model.predict(X_test)
r2_sample   = r2_score(y_test, y_pred_sample)
rmse_sample = np.sqrt(mean_squared_error(y_test, y_pred_sample))
mae_sample  = mean_absolute_error(y_test, y_pred_sample)

print("\n╔═════════ MODÈLE GAUSSIANPROCESSREGRESSOR ═════════╗")
print(f" ➤ Temps d'entraînement : {training_time_ms:.2f} ms")
print(" --- TRAIN (sous-ensemble X_tr) ---")
print(f"    R² TRAIN : {r2_tr:.4f}")
print(f"    MAE TRAIN : {mae_tr:.4f}")
print(f"    RMSE TRAIN: {rmse_tr:.4f}")
print(" --- VALIDATION interne (X_val) ---")
print(f"    R² VAL : {r2_val:.4f}")
print(f"    MAE VAL : {mae_val:.4f}")
print(f"    RMSE VAL: {rmse_val:.4f}")
print(" --- TEST (dates 5 %) ---")
print(f"    R² TEST : {r2_sample:.4f}")
print(f"    MAE TEST : {mae_sample:.4f}")
print(f"    RMSE TEST : {rmse_sample:.4f}")
print("╚══════════════════════════════════════════════════╝")

## 8. Évaluation cartographique (toutes les dates TEST) avec GPR

In [ ]:
# ------------------------------------------------------------
# 8. Évaluation cartographique (toutes les dates TEST) avec GPR
# ------------------------------------------------------------
print("\n=== ÉVALUATION CARTOGRAPHIQUE SUR TOUTES LES DATES TEST (5% des dates) ===")
region_mask = get_region_mask(template_profile)

all_true = []
all_pred = []

for date_key in test_dates:
    print(f"\n🧪 Test cartographique pour la date : {date_key} ")

    ndvi_date = get_ndvi_date_for(date_key)
    if ndvi_date is None:
        print(f" ⏭️ {date_key} : aucun NDVI disponible (<= {date_key}), ignoré.")
        continue

    et_date = get_et_date_for(date_key)
    if et_date is None:
        print(f" ⏭️ {date_key} : aucun ET disponible (<= {date_key}), ignoré.")
        continue

    if date_key not in gws_dict:
        print(f" ⏭️ {date_key} : GWS manquant, ignoré.")
        continue

    # GWS vérité terrain
    gws_raw, gws_prof, gws_nodata = read_raster_raw(gws_dict[date_key])
    gws_arr = to_float_with_nan(gws_raw, gws_nodata)

    # Entrées
    if (
        date_key not in chirps_dict
        or date_key not in lst_dict
        or date_key not in qsb_dict
        or date_key not in rzsm_dict
        or date_key not in soilm_dict
    ):
        print(
            f" ⏭️ {date_key} : au moins une variable journalière est manquante, ignorée."
        )
        continue

    precip_raw, precip_prof, precip_nodata = read_raster_raw(chirps_dict[date_key])
    lst_raw,    lst_prof,    lst_nodata    = read_raster_raw(lst_dict[date_key])
    ndvi_raw,   ndvi_prof,   ndvi_nodata   = read_raster_raw(ndvi_dict[ndvi_date])
    et_raw,     et_prof,     et_nodata     = read_raster_raw(et_dict[et_date])
    qsb_raw,    qsb_prof,    qsb_nodata    = read_raster_raw(qsb_dict[date_key])
    rzsm_raw,   rzsm_prof,   rzsm_nodata   = read_raster_raw(rzsm_dict[date_key])
    soilm_raw,  soilm_prof,  soilm_nodata  = read_raster_raw(soilm_dict[date_key])

    # Rééchantillonnage
    precip_res = reproject_to_template(
        precip_raw,
        precip_prof,
        template_profile,
        src_nodata=precip_nodata,
        dst_nodata=np.nan,
    )
    lst_res = reproject_to_template(
        lst_raw, lst_prof, template_profile, src_nodata=lst_nodata, dst_nodata=np.nan
    )
    ndvi_res = reproject_to_template(
        ndvi_raw, ndvi_prof, template_profile, src_nodata=ndvi_nodata, dst_nodata=np.nan
    )
    et_res = reproject_to_template(
        et_raw, et_prof, template_profile, src_nodata=et_nodata, dst_nodata=np.nan
    )
    qsb_res = reproject_to_template(
        qsb_raw, qsb_prof, template_profile, src_nodata=qsb_nodata, dst_nodata=np.nan
    )
    rzsm_res = reproject_to_template(
        rzsm_raw, rzsm_prof, template_profile, src_nodata=rzsm_nodata, dst_nodata=np.nan
    )
    soilm_res = reproject_to_template(
        soilm_raw,
        soilm_prof,
        template_profile,
        src_nodata=soilm_nodata,
        dst_nodata=np.nan,
    )

    # Remplissage NaN → cartes sans trous à l'intérieur des Regions
    precip_f = fill_nan_with_mean(precip_res)
    lst_f    = fill_nan_with_mean(lst_res)
    ndvi_f   = fill_nan_with_mean(ndvi_res)
    et_f     = fill_nan_with_mean(et_res)
    qsb_f    = fill_nan_with_mean(qsb_res)
    rzsm_f   = fill_nan_with_mean(rzsm_res)
    soilm_f  = fill_nan_with_mean(soilm_res)

    # DOY sin/cos pour la date courante
    doy = date_key.timetuple().tm_yday
    sin_doy = np.sin(2 * np.pi * doy / 365.0)
    cos_doy = np.cos(2 * np.pi * doy / 365.0)

    n_pixels_full = precip_f.size
    sin_plane_full = np.full(n_pixels_full, sin_doy, dtype="float32")
    cos_plane_full = np.full(n_pixels_full, cos_doy, dtype="float32")

    # Features sur tous les pixels
    X_all = np.stack(
        [
            precip_f.ravel(),
            lst_f.ravel(),
            ndvi_f.ravel(),
            et_f.ravel(),
            qsb_f.ravel(),
            rzsm_f.ravel(),
            soilm_f.ravel(),
            sin_plane_full,
            cos_plane_full,
        ],
        axis=1,
    )

    # 🔒 Sécurité : si X_all contient encore des NaN → on ignore cette date
    if np.isnan(X_all).any():
        print(
            f" ⏭️ {date_key} : X_all contient encore des NaN (features incomplètes), "
            "date ignorée pour l'évaluation cartographique."
        )
        continue

    y_pred_all = gpr_model.predict(X_all).astype("float32")
    pred_full = y_pred_all.reshape(
        template_profile["height"], template_profile["width"]
    )

    # Clip sur Regions
    pred_clip = pred_full.copy()
    pred_clip[~region_mask] = NODATA_VALUE

    # Masque pour les métriques : GWS défini + dans Regions
    mask_gws = (~np.isnan(gws_arr)) & region_mask
    if not np.any(mask_gws):
        print(" ⚠️ Aucun pixel valide dans GWS pour cette date, métriques ignorées.")
        continue

    y_true = gws_arr[mask_gws]
    y_pred = pred_clip[mask_gws]

    r2_d   = r2_score(y_true, y_pred)
    mae_d  = mean_absolute_error(y_true, y_pred)
    rmse_d = np.sqrt(mean_squared_error(y_true, y_pred))

    print(f" ➤ R² (carte clipée) : {r2_d:.4f}")
    print(f" ➤ MAE (carte clipée) : {mae_d:.4f}")
    print(f" ➤ RMSE (carte clipée) : {rmse_d:.4f}")

    all_true.append(y_true)
    all_pred.append(y_pred)

    # Sauvegarde carte de test
    out_profile_test = template_profile.copy()
    out_profile_test.update(dtype="float32", count=1, nodata=NODATA_VALUE)
    out_name_test = os.path.join(
        OUT_DIR,
        f"GWS_TEST_PRED_GPR_{date_key.strftime('%Y%m%d')}.tif",
    )
    with rasterio.open(out_name_test, "w", **out_profile_test) as dst:
        dst.write(pred_clip, 1)
    print(f" ✔️ Carte test prédite (clipée Regions) : {out_name_test}")

# Métriques globales (toutes dates TEST)
if len(all_true) > 0:
    all_true_flat = np.concatenate(all_true)
    all_pred_flat = np.concatenate(all_pred)
    r2_global   = r2_score(all_true_flat, all_pred_flat)
    mae_global  = mean_absolute_error(all_true_flat, all_pred_flat)
    rmse_global = np.sqrt(mean_squared_error(all_true_flat, all_pred_flat))

    print("\n📊 MÉTRIQUES GLOBALES (toutes dates TEST, cartes clipées Regions)")
    print(f" ➤ R² global : {r2_global:.4f}")
    print(f" ➤ MAE global : {mae_global:.4f}")
    print(f" ➤ RMSE global: {rmse_global:.4f}")
else:
    print("\n⚠️ Impossible de calculer des métriques globales (aucun pixel valide).")

## 9. Phase de PRÉDICTION CARTOGRAPHIQUE (FORECAST) avec GPR

In [ ]:
# ------------------------------------------------------------
# 9. Phase de PRÉDICTION CARTOGRAPHIQUE (FORECAST) avec GPR
# ------------------------------------------------------------
print("\n=== PHASE DE PRÉDICTION CARTOGRAPHIQUE (SANS TROUS, CLIPÉE REGIONS) ===")

if template_profile is None:
    raise RuntimeError("Template GLDAS introuvable (aucun GWS lu).")

out_profile = template_profile.copy()
out_profile.update(dtype="float32", count=1, nodata=NODATA_VALUE)

print("\n🗺️ Dossier de sortie des prédictions :")
print(" OUT_DIR =", OUT_DIR)

print("\n🗺️ Préparation de la liste des dates de prédiction...")

# Intersection des variables JOURNALIÈRES (sans NDVI / ET)
all_forecast_dates = sorted(
    set(chirps_dict.keys())
    & set(lst_dict.keys())
    & set(qsb_dict.keys())
    & set(rzsm_dict.keys())
    & set(soilm_dict.keys())
)

if len(all_forecast_dates) == 0:
    print(
        "⚠️ Aucune date commune CHIRPS ∩ LST ∩ Qsb ∩ RZSM ∩ SoilM. Impossible de prédire."
    )
else:
    print(
        "Plage possible (CHIRPS ∩ LST ∩ Qsb ∩ RZSM ∩ SoilM) :",
        min(all_forecast_dates),
        "→",
        max(all_forecast_dates),
    )

user_start = FORECAST_START.date()
user_end   = FORECAST_END.date()

if all_forecast_dates:
    forecast_start_date = max(user_start, min(all_forecast_dates))
    forecast_end_date   = min(user_end,   max(all_forecast_dates))
else:
    forecast_start_date = user_start
    forecast_end_date   = user_end

print("Fenêtre demandée par l'utilisateur :", user_start, "→", user_end)
print("Fenêtre réellement utilisée pour prédire:", forecast_start_date, "→", forecast_end_date)

forecast_dates = [
    d
    for d in all_forecast_dates
    if (forecast_start_date <= d <= forecast_end_date) and (d not in EXCLUDED_DATES)
]

print(
    f"\nNombre de dates éligibles (CHIRPS+LST+Qsb+RZSM+SoilM, hors dates exclues) : {len(forecast_dates)}"
)
if len(forecast_dates) == 0:
    print("⚠️ Aucune date éligible pour la prédiction même après recadrage.")
else:
    print("Première date de prédiction :", min(forecast_dates))
    print("Dernière date de prédiction :", max(forecast_dates))

print("\n🗺️ Génération des cartes prédites de GWS (SANS TROUS, CLIPÉES)...")
region_mask = get_region_mask(template_profile)

for date_key in forecast_dates:
    if date_key in EXCLUDED_DATES:
        print(f"⏭️ {date_key} : date exclue (LST manquante), ignorée.")
        continue

    if (date_key not in chirps_dict) or (date_key not in lst_dict):
        print(
            f"⏭️ {date_key} : CHIRPS ou LST manquants (cas inattendu), ignorée."
        )
        continue

    ndvi_date = get_ndvi_date_for(date_key)
    if ndvi_date is None:
        print(f"⏭️ {date_key} : aucun NDVI disponible (<= {date_key}), ignorée.")
        continue

    et_date = get_et_date_for(date_key)
    if et_date is None:
        print(f"⏭️ {date_key} : aucun ET disponible (<= {date_key}), ignorée.")
        continue

    # Lecture des entrées
    precip_raw, precip_prof, precip_nodata = read_raster_raw(chirps_dict[date_key])
    lst_raw,    lst_prof,    lst_nodata    = read_raster_raw(lst_dict[date_key])
    ndvi_raw,   ndvi_prof,   ndvi_nodata   = read_raster_raw(ndvi_dict[ndvi_date])
    et_raw,     et_prof,     et_nodata     = read_raster_raw(et_dict[et_date])
    qsb_raw,    qsb_prof,    qsb_nodata    = read_raster_raw(qsb_dict[date_key])
    rzsm_raw,   rzsm_prof,   rzsm_nodata   = read_raster_raw(rzsm_dict[date_key])
    soilm_raw,  soilm_prof,  soilm_nodata  = read_raster_raw(soilm_dict[date_key])

    # Rééchantillonnage
    precip_res = reproject_to_template(
        precip_raw,
        precip_prof,
        template_profile,
        src_nodata=precip_nodata,
        dst_nodata=np.nan,
    )
    lst_res = reproject_to_template(
        lst_raw, lst_prof, template_profile, src_nodata=lst_nodata, dst_nodata=np.nan
    )
    ndvi_res = reproject_to_template(
        ndvi_raw, ndvi_prof, template_profile, src_nodata=ndvi_nodata, dst_nodata=np.nan
    )
    et_res = reproject_to_template(
        et_raw, et_prof, template_profile, src_nodata=et_nodata, dst_nodata=np.nan
    )
    qsb_res = reproject_to_template(
        qsb_raw, qsb_prof, template_profile, src_nodata=qsb_nodata, dst_nodata=np.nan
    )
    rzsm_res = reproject_to_template(
        rzsm_raw, rzsm_prof, template_profile, src_nodata=rzsm_nodata, dst_nodata=np.nan
    )
    soilm_res = reproject_to_template(
        soilm_raw,
        soilm_prof,
        template_profile,
        src_nodata=soilm_nodata,
        dst_nodata=np.nan,
    )

    # Remplissage NaN
    precip_f = fill_nan_with_mean(precip_res)
    lst_f    = fill_nan_with_mean(lst_res)
    ndvi_f   = fill_nan_with_mean(ndvi_res)
    et_f     = fill_nan_with_mean(et_res)
    qsb_f    = fill_nan_with_mean(qsb_res)
    rzsm_f   = fill_nan_with_mean(rzsm_res)
    soilm_f  = fill_nan_with_mean(soilm_res)

    # DOY sin/cos pour la date de forecast
    doy = date_key.timetuple().tm_yday
    sin_doy = np.sin(2 * np.pi * doy / 365.0)
    cos_doy = np.cos(2 * np.pi * doy / 365.0)

    n_pixels_full = precip_f.size
    sin_plane_full = np.full(n_pixels_full, sin_doy, dtype="float32")
    cos_plane_full = np.full(n_pixels_full, cos_doy, dtype="float32")

    # Features sur tous les pixels
    X_all = np.stack(
        [
            precip_f.ravel(),
            lst_f.ravel(),
            ndvi_f.ravel(),
            et_f.ravel(),
            qsb_f.ravel(),
            rzsm_f.ravel(),
            soilm_f.ravel(),
            sin_plane_full,
            cos_plane_full,
        ],
        axis=1,
    )

    # 🔒 Sécurité : si X_all contient encore des NaN → on ignore cette date
    if np.isnan(X_all).any():
        print(
            f" ⏭️ {date_key} : X_all contient encore des NaN (features incomplètes), "
            "ignorée pour la prédiction."
        )
        continue

    y_pred_all = gpr_model.predict(X_all).astype("float32")
    pred_full = y_pred_all.reshape(
        template_profile["height"], template_profile["width"]
    )

    pred_clip = pred_full.copy()
    pred_clip[~region_mask] = NODATA_VALUE

    out_name = os.path.join(
        OUT_DIR,
        f"GWS_PRED_GPR_{date_key.strftime('%Y%m%d')}.tif",
    )
    with rasterio.open(out_name, "w", **out_profile) as dst:
        dst.write(pred_clip, 1)
    print(f" ✔️ Carte prédite (sans trous, clipée Regions) : {out_name}")

# Récapitulatif du dossier de sortie
if os.path.isdir(OUT_DIR):
    tif_files = [f for f in os.listdir(OUT_DIR) if f.lower().endswith(".tif")]
    print(f"\n📂 Nombre total de cartes GWS générées dans {OUT_DIR} : {len(tif_files)}")
    if len(tif_files) == 0:
        print("⚠️ Aucune image .tif trouvée dans le dossier de sortie.")
else:
    print(f"⚠️ Le dossier de sortie {OUT_DIR} n'existe pas.")

## Récapitulatif des sorties (ajout Colab)

In [ ]:
# [COLAB] Nombre de fichiers présents dans le dossier de sortie, par extension
from collections import Counter

_count = Counter(os.path.splitext(_f)[1].lower() or "(sans extension)" for _f in os.listdir(OUT_DIR))
for _ext, _n in sorted(_count.items()):
    print(f"{_n:6d} fichier(s) {_ext}")
print("Dossier de sortie :", OUT_DIR)